# Build Submissions From `inference_eval`

Локальный notebook для сборки `zip`-посылок из уже сохранённых `test_probs_*.pt` и `threshold_sweep_*.csv`.


In [1]:
import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch


In [2]:
ROOT = Path('.')
INFER_DIR = ROOT / 'inference_eval'
TEST_INFO_CSV = ROOT / 'info.csv'
OUT_DIR = ROOT / 'submissions_from_inference_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEV_H, BEV_W = 188, 126

best_probs_path = INFER_DIR / 'test_probs_best.pt'
ema_probs_candidates = [
    INFER_DIR / 'test_probs_ema_best.pt',
    INFER_DIR / 'test_probs_ema.pt',
]
ema_probs_path = next((p for p in ema_probs_candidates if p.exists()), None)

best_sweep_path = INFER_DIR / 'threshold_sweep_best.csv'
ema_sweep_candidates = [
    INFER_DIR / 'threshold_sweep_ema_best.csv',
    INFER_DIR / 'threshold_sweep_ema.csv',
]
ema_sweep_path = next((p for p in ema_sweep_candidates if p.exists()), None)

assert INFER_DIR.exists(), INFER_DIR
assert TEST_INFO_CSV.exists(), TEST_INFO_CSV
assert best_probs_path.exists(), best_probs_path
assert ema_probs_path is not None, 'No EMA test_probs file found'
assert best_sweep_path.exists(), best_sweep_path
assert ema_sweep_path is not None, 'No EMA threshold_sweep csv found'

print('best_probs_path:', best_probs_path)
print('ema_probs_path:', ema_probs_path)
print('best_sweep_path:', best_sweep_path)
print('ema_sweep_path:', ema_sweep_path)
print('test_info:', TEST_INFO_CSV)
print('out_dir:', OUT_DIR)


best_probs_path: inference_eval/test_probs_best.pt
ema_probs_path: inference_eval/test_probs_ema.pt
best_sweep_path: inference_eval/threshold_sweep_best.csv
ema_sweep_path: inference_eval/threshold_sweep_ema_best.csv
test_info: info.csv
out_dir: submissions_from_inference_eval


In [3]:
test_info = pd.read_csv(TEST_INFO_CSV, index_col=0).reset_index(drop=True)
print('test rows:', len(test_info))
display(test_info.head())

best_sweep = pd.read_csv(best_sweep_path)
ema_sweep = pd.read_csv(ema_sweep_path)

display(best_sweep.sort_values('iou', ascending=False).head(10))
display(ema_sweep.sort_values('iou', ascending=False).head(10))


test rows: 2000


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,...,/side/left/forward/car_to_cam,/side/right/forward/intrinsic_params,/side/right/forward/car_to_cam,/camera/inner/frontal/far/intrinsic_params,/camera/inner/frontal/far/car_to_cam,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order
0,autonomy_yandex_dataset_test/static_grids/1706...,1706096985200039000,14:22:32,1706096985200039000,autonomy_yandex_dataset_test/images/1706096985...,autonomy_yandex_dataset_test/images/1706096985...,autonomy_yandex_dataset_test/images/1706096985...,autonomy_yandex_dataset_test/images/1706096985...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/matrices/17060969...,...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/matrices/17060969...,autonomy_yandex_dataset_test/predicted_static_...,2024-01-24,13:21:16,natelio,0
1,autonomy_yandex_dataset_test/static_grids/1706...,1706097249999986000,14:22:32,1706097249999986000,autonomy_yandex_dataset_test/images/1706097249...,autonomy_yandex_dataset_test/images/1706097249...,autonomy_yandex_dataset_test/images/1706097249...,autonomy_yandex_dataset_test/images/1706097249...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/matrices/17060972...,...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/matrices/17060972...,autonomy_yandex_dataset_test/predicted_static_...,2024-01-24,13:21:16,natelio,0
2,autonomy_yandex_dataset_test/static_grids/1715...,1715156441999867000,11:16:12,1715156441999867000,autonomy_yandex_dataset_test/images/1715156441...,autonomy_yandex_dataset_test/images/1715156441...,autonomy_yandex_dataset_test/images/1715156441...,autonomy_yandex_dataset_test/images/1715156441...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/matrices/17151564...,...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/matrices/17151564...,autonomy_yandex_dataset_test/predicted_static_...,2024-05-08,11:16:12,hetera,0
3,autonomy_yandex_dataset_test/static_grids/1715...,1715161824799910000,12:43:44,1715161824799910000,autonomy_yandex_dataset_test/images/1715161824...,autonomy_yandex_dataset_test/images/1715161824...,autonomy_yandex_dataset_test/images/1715161824...,autonomy_yandex_dataset_test/images/1715161824...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/matrices/17151618...,...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/matrices/17151618...,autonomy_yandex_dataset_test/predicted_static_...,2024-05-08,12:43:44,natelio,0
4,autonomy_yandex_dataset_test/static_grids/1717...,1717418724299998000,15:34:54,1717418724299998000,autonomy_yandex_dataset_test/images/1717418724...,autonomy_yandex_dataset_test/images/1717418724...,autonomy_yandex_dataset_test/images/1717418724...,autonomy_yandex_dataset_test/images/1717418724...,autonomy_yandex_dataset_test/matrices/17174187...,autonomy_yandex_dataset_test/matrices/17174187...,...,autonomy_yandex_dataset_test/matrices/17174187...,autonomy_yandex_dataset_test/matrices/17174187...,autonomy_yandex_dataset_test/matrices/17174187...,autonomy_yandex_dataset_tes

,threshold,iou
18,0.56,0.604716
17,0.54,0.604621
16,0.52,0.604319
19,0.58,0.604288
15,0.50,0.603915
20,0.60,0.603556
14,0.48,0.603257
21,0.62,0.602884
13,0.46,0.602670
12,0.44,0.601924


,threshold,iou
17,0.54,0.603734
18,0.56,0.603562
16,0.52,0.603548
15,0.50,0.603219
19,0.58,0.603209
20,0.60,0.602735
14,0.48,0.602664
13,0.46,0.601951
21,0.62,0.601497
12,0.44,0.601072


In [4]:
def best_threshold_from_sweep(df: pd.DataFrame) -> float:
    row = df.sort_values(['iou', 'threshold'], ascending=[False, True]).iloc[0]
    return float(row['threshold'])


best_t_best = best_threshold_from_sweep(best_sweep)
best_t_ema = best_threshold_from_sweep(ema_sweep)

print('best threshold for best:', best_t_best)
print('best threshold for ema:', best_t_ema)


best threshold for best: 0.56
best threshold for ema: 0.54


In [5]:
def load_probs(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location='cpu')
    if isinstance(obj, dict) and 'probs' in obj:
        obj = obj['probs']
    if isinstance(obj, np.ndarray):
        obj = torch.from_numpy(obj)
    if not isinstance(obj, torch.Tensor):
        raise TypeError(f'Unexpected probs type: {type(obj)}')
    return obj.float()


best_probs = load_probs(best_probs_path)
ema_probs = load_probs(ema_probs_path)

print('best_probs shape:', tuple(best_probs.shape))
print('ema_probs shape:', tuple(ema_probs.shape))
assert len(best_probs) == len(test_info), (len(best_probs), len(test_info))
assert len(ema_probs) == len(test_info), (len(ema_probs), len(test_info))


best_probs shape: (2000, 1, 188, 126)
ema_probs shape: (2000, 1, 188, 126)


In [6]:
def pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'

import functools
from src.submission import make_submission_from_probs as _make_submission_from_probs
make_submission_from_probs = functools.partial(
    _make_submission_from_probs,
    out_root=OUT_DIR, test_info=test_info,
    info_csv=TEST_INFO_CSV, pred_name_fn=pred_name_from_row,
)


In [ ]:
best_thresholds = sorted(set([0.745, 0.75, 0.755, 0.765, 0.77, 0.775]))
best_thresholds = [t for t in best_thresholds if 0.05 <= t <= 0.95]

ema_thresholds = sorted(set([0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88]))
ema_thresholds = [t for t in ema_thresholds if 0.05 <= t <= 0.95]

print('best_thresholds:', best_thresholds)
print('ema_thresholds:', ema_thresholds)


best_thresholds: [0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88]
ema_thresholds: [0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88]


In [9]:
submission_results = []

for t in best_thresholds:
    print('\n' + '=' * 100)
    print(f'building BEST submission for threshold={t:.2f}')
    submission_results.append(make_submission_from_probs(best_probs, t, 'best'))

for t in ema_thresholds:
    print('\n' + '=' * 100)
    print(f'building EMA submission for threshold={t:.2f}')
    submission_results.append(make_submission_from_probs(ema_probs, t, 'ema'))

submission_results_df = pd.DataFrame(submission_results)
display(submission_results_df)



building BEST submission for threshold=0.50
{
  "model_tag": "best",
  "threshold": 0.5,
  "zip_path": "/Users/r-shangareev/PyProjects/shine-time/ML2_2026_Competition/submissions_from_inference_eval/submission_best_t_0p50.zip",
  "size_mb": 2.684,
  "sha256": "ffcd82abaa797675ebed9e2f9f460188b39e2677a790dc97df6f217aff0d7981"
}

building BEST submission for threshold=0.52
{
  "model_tag": "best",
  "threshold": 0.52,
  "zip_path": "/Users/r-shangareev/PyProjects/shine-time/ML2_2026_Competition/submissions_from_inference_eval/submission_best_t_0p52.zip",
  "size_mb": 2.729,
  "sha256": "f26a9d5c137acb651cdb7ab559b9f5bbc7765ece6c2b688ab402c057054b9a07"
}

building BEST submission for threshold=0.54
{
  "model_tag": "best",
  "threshold": 0.54,
  "zip_path": "/Users/r-shangareev/PyProjects/shine-time/ML2_2026_Competition/submissions_from_inference_eval/submission_best_t_0p54.zip",
  "size_mb": 2.777,
  "sha256": "2f96dc34531a1e20edaa41645a2efd27ef0fd5e351ee226c3ce8ee3616a7fe26"
}

buildin

,model_tag,threshold,zip_path,size_mb,sha256
0,best,0.50,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.684,ffcd82abaa797675ebed9e2f9f460188b39e2677a790dc...
1,best,0.52,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.729,f26a9d5c137acb651cdb7ab559b9f5bbc7765ece6c2b68...
2,best,0.54,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.777,2f96dc34531a1e20edaa41645a2efd27ef0fd5e351ee22...
3,best,0.56,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.830,5e284dc47a3b6ea5cf0c2667757b68771eb9862525253b...
4,best,0.58,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.883,55e41bbf5c5c84b64e6b7f1fda60df8bcd0e5cdfcc58ab...
5,best,0.60,/Users/r-shangareev/PyProjects/shine-time/ML2_...,2.938,334c85a5fb0200ec6a722f22ec15d19e3a24af63a21485...
6,best,0.62,/Users/r-shangareev/PyProjects/shine-time/ML2_...,3.001,8c01bd0925bac527951f08a67f76e3408898db4dc27c06...
7,best,0.64,/Users/r-shangareev/PyProjects/shine-time/ML2_...,3.063,564b6a921a5e5c9ad442df9cbda4fff5f3f2cfb233a642...
8,best,0.66,/Users/r-shangareev/PyProjects/shine-time/ML2_...,3.125,ccecba0a091cbc2dc33d938bdcd0b8798af15607c29e7f...
9,best,0.68,/Users/r-shangareev/PyProjects/shine-time/ML2_...,3.185,cb28bcde01fbba756e11bbcccbf300b662254016873b27...


In [ ]:
bundle_path = OUT_DIR / 'all_submissions_bundle.zip'
if bundle_path.exists():
    bundle_path.unlink()

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    manifest = []
    for row in submission_results:
        p = Path(row['zip_path'])
        zf.write(p, arcname=p.name)
        manifest.append(row)
    zf.writestr('manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

print('bundle:', bundle_path)
print('bundle size_mb:', round(bundle_path.stat().st_size / 1e6, 3))
